# Episode 11 — A web dashboard

The board runs Linux, so it can be a web server. In this episode a small Python server
shows the sensor readings on any phone or PC on the same network, and lets you switch
the LEDs and write on the 7-segment display from the browser.

```
phone browser  --HTTP-->  Python server on the ARM  --AXI-->  LEDs, switches, XADC in the PL
```

It uses only Python's built-in `http.server`, no extra packages. The page asks the server
for `/api` once a second and redraws the numbers.

**Note:** anyone on your network can open this page and control the board. Stop the server
when you are done.

In [ ]:
from pynq.overlays.base import BaseOverlay
import json, subprocess, threading, urllib.parse
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

base = BaseOverlay("base.bit")
PORT = 8000

## 1. The web page

Plain HTML with a little JavaScript: `fetch('/api')` gets the readings as JSON,
and each LED button calls `/led?n=<number>`.

In [ ]:
PAGE = """<!doctype html>
<html><head><meta name="viewport" content="width=device-width, initial-scale=1">
<title>EDGE ZYNQ</title>
<style>
 body { font-family: sans-serif; margin: 16px; background: #f4f6f8; color: #222; }
 .tiles { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 10px; }
 .tile { background: #fff; border-radius: 8px; padding: 12px; box-shadow: 0 1px 3px #0002; }
 .tile b { display: block; font-size: 28px; margin-top: 4px; }
 button { font-size: 18px; margin: 4px; padding: 10px 14px; border-radius: 6px; border: 1px solid #888; }
 button.on { background: #ffd54a; }
 input { font-size: 18px; width: 6em; padding: 8px; }
</style></head>
<body>
<h2>EDGE ZYNQ dashboard</h2>
<div class="tiles">
 <div class="tile">Zynq chip<b id="die">-</b></div>
 <div class="tile">LM35<b id="lm35">-</b></div>
 <div class="tile">Light<b id="light">-</b></div>
 <div class="tile">Switches<b id="sw">-</b></div>
</div>
<h3>LEDs</h3>
<div id="leds"></div>
<h3>7-segment display</h3>
<input id="text" maxlength="4" value="HI"> <button onclick="show()">Show</button>
<script>
const leds = document.getElementById('leds');
for (let i = 0; i < 4; i++) {
  const b = document.createElement('button');
  b.textContent = 'LED ' + i; b.id = 'led' + i;
  b.onclick = () => fetch('/led?n=' + i).then(update);
  leds.appendChild(b);
}
function show() {
  fetch('/show?text=' + encodeURIComponent(document.getElementById('text').value));
}
function update() {
  fetch('/api').then(r => r.json()).then(d => {
    document.getElementById('die').textContent = d.die_c.toFixed(1) + ' °C';
    document.getElementById('lm35').textContent = d.lm35_c.toFixed(1) + ' °C';
    document.getElementById('light').textContent = d.light_v.toFixed(2) + ' V';
    document.getElementById('sw').textContent = d.switches;
    for (let i = 0; i < 4; i++)
      document.getElementById('led' + i).className = (d.leds >> i) & 1 ? 'on' : '';
  });
}
setInterval(update, 1000); update();
</script></body></html>"""

## 2. The server

`do_GET` runs for every request. Each request reads or writes the board directly.
The LED state is kept in Python because the LEDs are outputs only.

In [ ]:
led_state = 0
lock = threading.Lock()          # requests arrive on several threads at once

def readings():
    return {"die_c": base.xadc.temperature(),
            "lm35_c": base.xadc.lm35_celsius(),
            "light_v": base.xadc.light_level(),
            "switches": format(base.switches.read(), "04b"),
            "leds": led_state}

class Handler(BaseHTTPRequestHandler):
    def reply(self, body, kind="application/json", code=200):
        data = body.encode() if isinstance(body, str) else body
        self.send_response(code)
        self.send_header("Content-Type", kind)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        global led_state
        url = urllib.parse.urlparse(self.path)
        query = urllib.parse.parse_qs(url.query)
        with lock:
            if url.path == "/":
                self.reply(PAGE, "text/html; charset=utf-8")
            elif url.path == "/api":
                self.reply(json.dumps(readings()))
            elif url.path == "/led" and query.get("n", [""])[0] in ("0", "1", "2", "3"):
                led_state ^= 1 << int(query["n"][0])
                base.leds[0:4].write(led_state)
                self.reply(json.dumps({"leds": led_state}))
            elif url.path == "/show":
                text = query.get("text", [""])[0][:4]
                base.seven_seg.show_text(text)
                self.reply(json.dumps({"shown": text}))
            else:
                self.reply('{"error": "not found"}', code=404)

    def log_message(self, *args):     # keep the notebook quiet
        pass

ThreadingHTTPServer.allow_reuse_address = True
server = ThreadingHTTPServer(("", PORT), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()

ip = subprocess.check_output(["hostname", "-I"]).decode().split()[0]
print("Open http://{}:{} on your phone or PC".format(ip, PORT))

## 3. Test it from the notebook

The notebook can be a client of its own server.

In [ ]:
import urllib.request

def get(path):
    with urllib.request.urlopen("http://localhost:{}{}".format(PORT, path), timeout=5) as r:
        return json.loads(r.read())

print(get("/api"))
print(get("/led?n=0"))
print(get("/show?text=COOL"))

## 4. Stop the server

In [ ]:
server.shutdown()
server.server_close()
print("Server stopped")

## Try it yourself

Add a `/beep` address that sounds the buzzer for 0.2 seconds, and a **Beep** button
on the page that calls it.

In [ ]:
# your code here